# 모두의 리서치 · A기업 주피터 실습
산업·사업모델 분석과 기술적 분석을 같은 보고서 버전에서 살펴봅니다.

**교육용 합성 자료 · 읽기 전용 탐색 · API 키/DB/웹 서버 불필요**

ZIP을 먼저 압축 해제하고 이 노트북을 엽니다. 아래 셀을 위에서부터 실행하세요.
최초 한 번 Jupyter 환경에 `ipywidgets`가 필요합니다. 위젯 설치가 어려우면 정적 보기 셀을 사용하세요.


In [ ]:
# 필요한 경우에만 주석을 해제하여 현재 노트북 커널에 설치합니다.
# %pip install "ipywidgets>=8,<9"


In [ ]:
import json
import sys
from pathlib import Path

# 프로젝트 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "agent/notebook_viewer.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("압축 해제한 프로젝트 폴더에서 노트북을 열어주세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from agent.notebook_viewer import load_reports, render_report, show_viewer
from agent.research_contract import revise_claim, validate_report

reports = load_reports()
print("A기업 예시 로드·참조·digest 검증 완료:", list(reports))


## 1. 대화형 화면
보고서 1판/2판을 고르고 산업·기술적 주장 버튼을 누릅니다.
근거의 **전체 원문 펼치기**를 열면 로컬 예시 원문이 보입니다.
2판에서는 변경된 C01의 근거가 비워지고, 과거 반론이 원래 버전의 이력에 남습니다.
위젯이 보이지 않으면 설치 후 커널을 다시 시작하고 위에서부터 실행하거나 다음 정적 보기를 사용하세요.


In [ ]:
viewer = show_viewer()


## 2. 정적 보기 · 위젯 없이도 가능
아래 `VERSION`과 `CLAIM`을 바꿔 다시 실행하세요. 원문 펼치기는 HTML 기본 기능입니다.


In [ ]:
from IPython.display import HTML, display

VERSION = 2  # 1 또는 2
CLAIM = "C01"  # 산업: C01 / 기술: C02
display(HTML(render_report(reports[VERSION], CLAIM)))


## 3. 공통 JSON 구조 확인
원자료·주장·근거·계산·반론은 LLM 호출 없이 로컬 JSON에서 가져옵니다.


In [ ]:
report = reports[1]
print(json.dumps({key: report[key] for key in
    ("reportId", "contentRevision", "dataMode", "claims", "analysisRelations")},
    ensure_ascii=False, indent=2))


## 4. 수정 버전 생성 실습
메모리에서만 수정합니다. 원본 JSON 파일은 바꾸지 않습니다.
이전 근거·계산 연결을 자동 승계하지 않으며 재검토가 필요합니다.


In [ ]:
revised = revise_claim(reports[1], "C01",
    text="제품 비중 변화는 확인 대상이며 비용·마진 검토 전 이익 개선은 보류한다.",
    reason="주피터에서 반론에 따른 문구 수정 실습")
validate_report(revised)
assert reports[1]["contentRevision"] == 1
assert revised["contentRevision"] == 2
assert revised["claims"][0]["evidenceLinks"] == []
assert revised["approvals"] == []
display(HTML(render_report(revised, "C01")))


## 범위와 다음 단계
이 노트북은 예시 JSON 조회·버전 실습 도구입니다. 실제 시장 데이터·AI 모델의 정확도·공유 권한을 검증하지 않습니다.
기존 페르소나·Solar Agent는 전체 OpenResearch 패키지에서 별도로 연결할 수 있습니다.
이 휴대용 노트북 패키지에는 그 실행에 필요한 API 키나 외부 서비스 설정을 넣지 않았습니다.
